# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# Install Python dependencies (run once, then skip)
import sys
!{sys.executable} -m pip install -q sympy numpy tqdm requests \
    'antlr4-python3-runtime==4.11.1' ipykernel

ERROR: Invalid requirement: "'antlr4-python3-runtime==4.11.1'": Expected package name at the start of dependency specifier
    'antlr4-python3-runtime==4.11.1'
    ^


### Run the cell below every time to activate the installed environment. 

In [2]:
# (No venv activation needed — packages installed directly into this kernel)
print('Environment ready.')

Environment ready.


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import re
import sys
from pathlib import Path
from typing import Optional

import requests
from tqdm import tqdm

# ── Configuration ────────────────────────────────────────────────────────────
LM_STUDIO_URL   = "http://localhost:1234"
LM_STUDIO_MODEL = "qwen/qwen3-4b"
DATA_PATH       = "data/public.jsonl"
OUTPUT_PATH     = "results/starter_results.jsonl"
MAX_TOKENS      = 8192

print(f"LM Studio URL   : {LM_STUDIO_URL}")
print(f"LM Studio model : {LM_STUDIO_MODEL}")
print(f"Max tokens      : {MAX_TOKENS}")

LM Studio URL   : http://localhost:1234
LM Studio model : qwen/qwen3-4b
Max tokens      : 8192


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Your final answer must appear in a SINGLE \\boxed{} expression. "
    "For multi-part problems, place ALL answers comma-separated inside ONE \\boxed{}, "
    "in the order asked — e.g. \\boxed{3, 7} or \\boxed{62.78, 335.93, 604.67}. "
    "Do NOT produce multiple separate \\boxed{} expressions. "
    "Do not round intermediate or final answers. Give exact values: use fractions (5/8), "
    "exact decimals, or symbolic expressions (e.g. atan(4.76), ln(0.5)/ln(0.96584)). "
    "If the problem asks for a formula or expression, give the expression — not a numeric approximation. "
    "When the answer is an expression, show all arithmetic steps in your answer — "
    "e.g. write 2*8*x rather than 16x if that matches the structure of the original problem."

)

SYSTEM_PROMPT_MCQ = (
     "You are an expert mathematician. "
    "Read the problem and the answer choices carefully, then select the single best answer. "
    "First work through the problem step-by-step to derive the answer. "
    "Then compare your result numerically or symbolically against each option. "
    "If your derived answer does not obviously match an option, substitute a concrete "
    "numeric value into both your answer and each option to find the closest match. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "You MUST always pick one of the given letters — never say none match."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with Transformers

We load **Qwen3-4B-Thinking-2507** using HuggingFace Transformers with automatic device placement.
- If a GPU is available it will be used automatically.
- On CPU this will be slow (~1-5 min per question) but will work.
- `torch_dtype=torch.float16` halves memory usage vs float32.


In [6]:
# Verify LM Studio is reachable and the model is loaded
try:
    r = requests.get(f"{LM_STUDIO_URL}/v1/models", timeout=5)
    r.raise_for_status()
    models = [m["id"] for m in r.json().get("data", [])]
    if models:
        print(f"LM Studio is running. Loaded models: {models}")
        if LM_STUDIO_MODEL not in models:
            print(f"WARNING: '{LM_STUDIO_MODEL}' not found. Update LM_STUDIO_MODEL to one of: {models}")
    else:
        print("WARNING: LM Studio is running but no model is loaded. Load a model in the Developer tab.")
except requests.exceptions.ConnectionError:
    print(f"ERROR: Cannot reach LM Studio at {LM_STUDIO_URL}")
    print("Make sure LM Studio is open and the server is started (Developer tab → Start Server)")

LM Studio is running. Loaded models: ['qwen/qwen3-4b', 'text-embedding-nomic-embed-text-v1.5']


## 6. Generate Responses

We process each question individually using `model.generate()`.
A progress bar shows estimated time remaining.
Adjust `data[:5]` to run on more questions.


In [ ]:
def generate_response(question: str, options=None) -> str:
    system, user = build_prompt(question, options)
    r = requests.post(
        f"{LM_STUDIO_URL}/v1/chat/completions",
        json={
            "model": LM_STUDIO_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            "temperature": 0.2,
            "top_p":       0.85,
            "top_k":       20,
            "max_tokens":  MAX_TOKENS,
        },
        timeout=600,
    )
    if not r.ok:
        raise RuntimeError(f"LM Studio error {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]


# Run on first 5 questions — change data[:5] to data[:] for the full dataset
sys.path.insert(0, '.')
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''

def score_response(item: dict, response: str) -> bool:
    if item.get('options'):
        return extract_letter(response) == item['answer'].strip().upper()
    gold_list = item['answer'] if isinstance(item['answer'], list) else [item['answer']]
    return judger.auto_judge(response, gold_list, options=[[]]*len(gold_list))

# ── Resume + Run ──────────────────────────────────────────────────────────────
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

done_ids = set()
results  = []
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            rec = json.loads(line)
            done_ids.add(rec['id'])
            results.append(rec)
    print(f"Resuming: {len(done_ids)} done, {len(data) - len(done_ids)} remaining")
else:
    print("Starting fresh run")

with open(out_path, 'a') as f:
    for item in tqdm(data, desc="Generating"):
        if item['id'] in done_ids:
            continue
        try:
            response = generate_response(item["question"], item.get("options"))
            correct  = score_response(item, response)
        except Exception as e:
            print(f"\nERROR on id={item['id']}: {e} — skipping")
            continue
        record = {
            'id':      item['id'],
            'is_mcq':  bool(item.get('options')),
            'gold':    item['answer'],
            'response': response,
            'correct':  correct,
        }
        results.append(record)
        f.write(json.dumps(record) + '\n')
        f.flush()

Resuming: 10 done, 1116 remaining


Generating:   0%|          | 0/1126 [00:00<?, ?it/s]

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, '.')
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in zip(subset, responses):
    is_mcq = bool(item.get('options'))
    gold   = item['answer']

    if is_mcq:
        correct = score_mcq(response, gold)
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        correct = judger.auto_judge(
            response,
            gold_list,
            options=[[]]*len(gold_list),
        )

    results.append({
        'id':       item['id'],
        'is_mcq':   is_mcq,
        'gold':     gold,
        'response': response,
        'correct':  correct,
    })
    print(f"id={item['id']} correct={correct}")


id=0 correct=True
id=1 correct=False
id=2 correct=False
id=3 correct=True
id=4 correct=True
id=5 correct=False
id=6 correct=True
id=7 correct=True
id=8 correct=False
id=9 correct=True


## 8. Summary

Print accuracy broken down by question type.

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    2 /    3  (66.67%)
  Free-form  :    4 /    7  (57.14%)
  Overall    :    6 /   10  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to results\starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!